<!-- nav -->
[← 8.15 · SystemVerilog Assertions](08_15_SystemVerilog_Assertions.ipynb) · [Home](../../README.md) · [8.17 · SystemVerilog Preprocessor →](08_17_SystemVerilog_Preprocessor.ipynb)

# 8.16 · SystemVerilog Constraints

A SystemVerilog testbench draws random stimulus, and says which values a randomized object may take in constraints
(IEEE 1800 18.5): `x inside {[0:9]}`, `x dist {0 := 1, [1:9] :/ 9}`, `mode == 1 -> len < 8`. Tutorial 8 showed two,
held by `randomize() with` (08.4). This page builds them up in their dialect, `SystemVerilog Constraints`: an expression
held as a constraint, `soft` or not, and a block of them; membership and weighted distributions; implications and
conditionals; a `foreach`, which binds its loop variables; the order, softness and uniqueness constraints. Each is
written in the builders' fluent calls beside the text SystemVerilog writes. Then what is done with them: held by
`randomize() with`, validated, with one refused; never evaluated, since no solver is modeled; and the editions that
have each kind.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `name` and
`literal` write a name and a literal of SystemVerilog's `Expressions` (tutorials 3 and 4), and `refused` gives what an
evaluator says of an expression, its value or the reason it refuses it, as tutorial 8's does.

In [1]:
from mbse.Expressions.SystemVerilog import Constraints as SVConstraints, Expressions as SystemVerilog
from mbse.Expressions.Framework import Domains, Standards, Symbolics, Terms
from toolkit import dialects

store = dialects()
S = store.Expressions
Constraints = SVConstraints.Expressions.DIALECT  # the dialect, as the framework reads it
name = lambda spelling: lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling(spelling))))
literal = lambda value: lambda d: d.literal(lambda l: l.value(lambda v: v.as_scalar(value)))
x, y = name("x"), name("y")



def refused(evaluate, expression):
    """What `evaluate` says of `expression`: its value, or the reason it refuses it."""
    try:
        return str(evaluate(expression))
    except (NotImplementedError, ValueError) as error:
        return str(error)

<a id="T08.16.1"></a>
## 08.16.1 · An expression held as a constraint

```text
x > 0;        soft x > 0;        { x > 0; y < x; }
```

The simplest constraint is an expression that must be true: an `expressionconstraint`, its `expression` a term of
`Expressions`, and `soft` if a solver may drop it when it conflicts with another (18.5.13). A `constraintblock` holds
constraints in braces, as a class's constraint block does. Each is a kind of `SystemVerilog Constraints`, the root of
its holder, `S.SystemVerilog.Constraints.Expression`, and the family's printer writes each constraint ended by its `;`:

In [2]:
with store.transaction():
    simple = [S.SystemVerilog.Constraints.Expression(lambda e, root=root: e.root(root)) for root in [
        lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.binaryexpression(lambda b: b.left(x).operator(">").right(literal(0))))),
        lambda c: c.expressionconstraint(lambda e: e.soft(True).expression(lambda d: d.binaryexpression(lambda b: b.left(x).operator(">").right(literal(0))))),
        lambda c: c.constraintblock(lambda b: b.items([lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.binaryexpression(lambda b: b.left(x).operator(">").right(literal(0))))), lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.binaryexpression(lambda b: b.left(y).operator("<").right(x))))])),
    ]]
for constraint in simple:
    form = Terms.form(Terms.term(constraint))
    print(f"{SystemVerilog.Text.ToText(constraint):19}", f"{form.kind:20}", "| its terms:",
          ", ".join(Terms.kind_of(t)[0] for t in form.arguments), "| free:", ", ".join(sorted(Symbolics.free(constraint, Constraints))))
assert [SystemVerilog.Text.ToText(c) for c in simple] == ["x > 0;", "soft x > 0;", "{ x > 0; y < x; }"]

x > 0;              expressionconstraint | its terms: binaryexpression | free: x
soft x > 0;         expressionconstraint | its terms: binaryexpression | free: x
{ x > 0; y < x; }   constraintblock      | its terms: expressionconstraint, expressionconstraint | free: x, y


<a id="T08.16.2"></a>
## 08.16.2 · Membership and distributions

```text
x inside {[0:9]}        x dist {0 := 40, [1:3] :/ 60, default :/ 1}
```

`x inside {[0:9]}` is an expression, `Expressions`' own `insideexpression`, whose set holds a range, `[0:9]`, a
`valuerange`; held as a constraint, it says `x` is among them. A distribution is this dialect's `distexpression`: its
value, an expression, and its items, each a `DistItem` helper holding a value or a range, and an operator and a weight:
`:=` gives each value in it the weight, `:/` divides the weight among them, and an item with no value is `default`
(IEEE 1800-2023):

In [3]:
inside = lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.insideexpression(lambda i: i.value(x).set([lambda d: d.valuerange(lambda r: r.left(literal(0)).right(literal(9)))]))))
with store.transaction():
    membership = [S.SystemVerilog.Constraints.Expression(lambda e, root=root: e.root(root)) for root in [
        inside,
        lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.distexpression(lambda t: t.value(x).items([
            lambda i: i.value(literal(0)).operator(":=").weight(literal(40)), lambda i: i.value(lambda d: d.valuerange(lambda r: r.left(literal(1)).right(literal(3)))).operator(":/").weight(literal(60)),
            lambda i: i.operator(":/").weight(literal(1))])))),
    ]]
for constraint in membership:
    expression = Terms.form(Terms.term(constraint)).arguments[0]
    form = Terms.form(expression)
    print(f"{SystemVerilog.Text.ToText(constraint):45}", form.kind, "of", Constraints.owner(expression).name())
    print(" ", ", ".join(f"{key}: {', '.join(getattr(p, 'kind', str(p)) for p in (value if isinstance(value, tuple) else (value,)))}"
                         for key, value in form.attributes.items()) or "-")
assert [SystemVerilog.Text.ToText(c) for c in membership] == [
    "x inside {[0:9]};", "x dist {0 := 40, [1:3] :/ 60, default :/ 1};"]

x inside {[0:9]};                             insideexpression of SystemVerilog
  -
x dist {0 := 40, [1:3] :/ 60, default :/ 1};  distexpression of SystemVerilog Constraints
  items: distitem, distitem, distitem


An `inside` belongs to `Expressions`, which this dialect's `Term` lists (tutorial 8, 08.3); a `dist` to this dialect,
and Assertions' `Term` lists it too, since a sequence's expression may be a distribution.

<a id="T08.16.3"></a>
## 08.16.3 · Implications and conditionals

```text
mode == 1 -> len < 8;        if (big) len > 8; else len <= 8;
```

An `implicationconstraint` holds its body where its condition is true; a `conditionalconstraint`, `if`, its consequence
where its condition is true and its alternative, which may be absent, where it is false. Their conditions are
expressions, `Expressions`' terms; their bodies are constraints, this dialect's own, a block among them:

In [4]:
mode, len_, big = name("mode"), name("len"), name("big")
implication = lambda c: c.implicationconstraint(lambda i: i.condition(lambda d: d.binaryexpression(lambda b: b.left(mode).operator("==").right(literal(1)))).body(lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.binaryexpression(lambda b: b.left(len_).operator("<").right(literal(8)))))))
with store.transaction():
    choices = [S.SystemVerilog.Constraints.Expression(lambda e, root=root: e.root(root)) for root in [
        implication,
        lambda c: c.conditionalconstraint(lambda i: i.condition(big)
            .consequence(lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.binaryexpression(lambda b: b.left(len_).operator(">").right(literal(8)))))).alternative(lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.binaryexpression(lambda b: b.left(len_).operator("<=").right(literal(8))))))),
    ]]
for constraint in choices:
    form = Terms.form(Terms.term(constraint))
    print(f"{SystemVerilog.Text.ToText(constraint):33}", f"{form.kind:22}",
          ", ".join(f"{Terms.kind_of(t)[0]} of {Constraints.owner(t).name()}" for t in form.arguments))
assert [SystemVerilog.Text.ToText(c) for c in choices] == ["mode == 1 -> len < 8;", "if (big) len > 8; else len <= 8;"]

mode == 1 -> len < 8;             implicationconstraint  binaryexpression of SystemVerilog, expressionconstraint of SystemVerilog Constraints
if (big) len > 8; else len <= 8;  conditionalconstraint  nameexpression of SystemVerilog, expressionconstraint of SystemVerilog Constraints, expressionconstraint of SystemVerilog Constraints


<a id="T08.16.4"></a>
## 08.16.4 · `foreach` binds its loop variables

```text
foreach (a[i]) a[i] < 10;
```

A `foreach` constrains each element of an array: it binds its loop variables, `i`, the index of each dimension, within
its body alone (18.5.8.1). So it is a binder, a variation of the framework's `let` (tutorial 7): it refines `Let`, its
array Let's `value` and its body Let's `body`, and leaves Let's `name` unset, since it binds a list of variables, which
its kind's scopes give. The body reads `i`, which the `foreach` gives it, and `a`, which the scope must:

In [5]:
loop = lambda c: c.foreachconstraint(lambda f: f.value(name("a"))
    .variables([lambda v: v.identifier(lambda i: i.spelling("i"))])
    .body(lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.binaryexpression(lambda b: b.left(lambda d: d.indexexpression(lambda e: e.value(name("a")).index(name("i")))).operator("<").right(literal(10)))))))
with store.transaction():
    each = S.SystemVerilog.Constraints.Expression(lambda e: e.root(loop))
form = Terms.form(Terms.term(each))
schema = Terms.kind_of(Terms.term(each))[1].as_schema()
print(SystemVerilog.Text.ToText(each), "|", form.kind, "refines", schema.refines.name, "| a binder:", "yes" if Terms.is_binder(schema) else "no")
print("binds", ", ".join(sorted(SVConstraints.Expressions.loop_variables(Terms.kind_of(Terms.term(each))[1]))),
      "| value:", SystemVerilog.Text.ToText(form.arguments[0]), "| body:", SystemVerilog.Text.ToText(form.arguments[1]),
      "| free:", ", ".join(sorted(Symbolics.free(each, Constraints))))
assert SystemVerilog.Text.ToText(each) == "foreach (a[i]) a[i] < 10;" and Symbolics.free(each, Constraints) == {"a"}

foreach (a[i]) a[i] < 10; | foreachconstraint refines Expressions.Framework.Let | a binder: yes
binds i | value: a | body: a[i] < 10; | free: a


Tutorial 7's list of every binder, read from the store, showed it under `SystemVerilog.Constraints`: the one binder of
SystemVerilog's sublanguages.

<a id="T08.16.5"></a>
## 08.16.5 · Order, softness and uniqueness

```text
solve x before y;        disable soft x;        unique {a, b, c};
```

Three constraints say how the others are solved. `solve ... before` orders the variables a solver picks values for
(18.5.10): it holds two lists of expressions, `solve` and `before`, which its form gives as two parts, each the number of
its terms. `disable soft` drops the soft constraints on its target; `unique` says its expressions, and ranges, have
values different from each other (both IEEE 1800-2012):

In [6]:
with store.transaction():
    order = [S.SystemVerilog.Constraints.Expression(lambda e, root=root: e.root(root)) for root in [
        lambda c: c.solvebeforeconstraint(lambda s: s.solve([x]).before([y])),
        lambda c: c.disablesoftconstraint(lambda d: d.target(x)),
        lambda c: c.uniqueconstraint(lambda u: u.set([name("a"), name("b"), name("c")])),
    ]]
for constraint in order:
    form = Terms.form(Terms.term(constraint))
    print(f"{SystemVerilog.Text.ToText(constraint):18}", f"{form.kind:22}",
          ", ".join(f"{key} of {part.size}" for key, part in form.attributes.items()) or "-", "| terms:", len(form.arguments))
assert [SystemVerilog.Text.ToText(c) for c in order] == ["solve x before y;", "disable soft x;", "unique {a, b, c};"]

solve x before y;  solvebeforeconstraint  solve of 1, before of 1 | terms: 2
disable soft x;    disablesoftconstraint  - | terms: 1
unique {a, b, c};  uniqueconstraint       - | terms: 3


<a id="T08.16.6"></a>
## 08.16.6 · Held by `randomize() with`, validated, and one refused

```text
p.randomize() with { x inside {[0:9]}; mode == 1 -> len < 8; foreach (a[i]) a[i] < 10; }
```

A call of `randomize() with` holds its constraints in its `items`, the one property of `Expressions`' kinds that holds
this dialect's `Term`, read from the store. Its host validates the tree, every constraint's names among them, and
`validate` checks each kind's own rules: a distribution's `default` item divides its weight among the values no other
item names, so it weighs them with `:/`, and one with `:=` is refused:

In [7]:
union = store.registered(Constraints.union)
host = store.registered(SystemVerilog.Expressions.DIALECT.union).branches
print(", ".join(f"{b.name}.{p}" for b in host for p, prop in b.schema.properties.items()
                if union in (prop.schema, getattr(prop.schema, "element", None))))
with store.transaction():
    randomized = S.SystemVerilog.Expressions.Expression(lambda e: e.root(lambda u: u.randomizewithexpression(lambda r: r
        .call(lambda c: c.callexpression(lambda m: m.callee(lambda d: d.memberexpression(lambda b: b
            .value(name("p")).member(lambda i: i.spelling("randomize")))).arguments([])))
        .items([inside, implication, loop]))))
dialect = SystemVerilog.Expressions.DIALECT
print(SystemVerilog.Text.ToText(randomized))
for bound in ({"p", "x", "mode"}, {"p", "x", "mode", "len", "a"}):
    print(" ", ", ".join(sorted(bound)), "|", "; ".join(dialect.validate(randomized, bound)) or "valid")
with store.transaction():
    fallback = S.SystemVerilog.Constraints.Expression(lambda e: e.root(lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.distexpression(lambda t: t.value(x).items([lambda i: i.operator(":=").weight(literal(1))]))))))
print(SystemVerilog.Text.ToText(fallback), "|", "; ".join(Constraints.validate(fallback, {"x"})))
assert Constraints.validate(fallback, {"x"}) == ["expression: a default DistItem weighs its values with :/"]

randomizewithexpression.items
p.randomize() with { x inside {[0:9]}; mode == 1 -> len < 8; foreach (a[i]) a[i] < 10; }
  mode, p, x | argument 1: body: expression: left: nameexpression 'len' is not bound; argument 2: value: nameexpression 'a' is not bound; argument 2: body: expression: left: value: nameexpression 'a' is not bound
  a, len, mode, p, x | valid
x dist {default := 1}; | expression: a default DistItem weighs its values with :/


<a id="T08.16.7"></a>
## 08.16.7 · Never evaluated

A constraint is solved, not evaluated, and no solver is modeled: SystemVerilog's evaluator refuses each of the
dialect's kinds by name (stage 6, "Q11"), held alone or by `randomize() with`, and a constraint has no domain to infer,
`any`:

In [8]:
evaluate = SystemVerilog.Evaluators.OfAny
for constraint in [simple[0], membership[1], choices[0], each, order[0]]:
    print(f"{SystemVerilog.Text.ToText(constraint):45}", refused(lambda e: evaluate(e, {"x": 1}), constraint))
print("inferred:", Constraints.infer(simple[0], {"x": Domains.Anything}).name())

x > 0;                                        SystemVerilog Constraints expressionconstraint is not evaluated
x dist {0 := 40, [1:3] :/ 60, default :/ 1};  SystemVerilog Constraints expressionconstraint is not evaluated
mode == 1 -> len < 8;                         SystemVerilog Constraints implicationconstraint is not evaluated
foreach (a[i]) a[i] < 10;                     SystemVerilog Constraints foreachconstraint is not evaluated
solve x before y;                             SystemVerilog Constraints solvebeforeconstraint is not evaluated
inferred: any


<a id="T08.16.8"></a>
## 08.16.8 · Its editions are data

Constraints arrived with SystemVerilog, so the dialect covers IEEE 1800's five editions and none of Verilog's, each with
the kinds it has
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):
IEEE 1800-2012 added `disable soft` and `unique`:

In [9]:
singleton = store.Instances.Expressions.SystemVerilog.Constraints
kinds = {branch.schema: branch.name for branch in store.registered(Constraints.union).branches}
for designation, edition in Standards.standards(singleton).items():
    print(f"{designation:19}", f"{Standards.tags(edition)[1]:19}", len(Standards.kinds(edition)), "kinds")
first, second = (Standards.kinds(Standards.find(singleton, label)) for label in ("SystemVerilog-2009", "SystemVerilog-2012"))
print("added in IEEE 1800-2012:", ", ".join(kinds[k] for k in second if k not in first))

IEEE Std 1800-2005  SystemVerilog-2005  7 kinds
IEEE Std 1800-2009  SystemVerilog-2009  7 kinds
IEEE Std 1800-2012  SystemVerilog-2012  9 kinds
IEEE Std 1800-2017  SystemVerilog-2017  9 kinds
IEEE Std 1800-2023  SystemVerilog-2023  9 kinds
added in IEEE 1800-2012: disablesoftconstraint, uniqueconstraint


An edition records the kinds it has, not each of a kind's forms: a distribution's `default` item is IEEE 1800-2023's, but
`distexpression` is in every edition's.

## What you learned

- **A constraint is a term of `SystemVerilog Constraints`**: an expression held as one, `soft` or not, a block, a
  distribution, an implication or a conditional, a `foreach`, and the order, softness and uniqueness constraints; their
  conditions and values are `Expressions`' terms.
- **`foreach` binds its loop variables**, a variation of `let`; nothing else in the dialect binds.
- **`randomize() with` holds them**, and its host validates the tree; a `default` item weighed with `:=` is refused.
- **No solver is modeled**: each kind is refused by name; its editions are IEEE 1800's, two kinds since IEEE 1800-2012.

Next, the directives and macros among them: [SystemVerilog Preprocessor](08_17_SystemVerilog_Preprocessor.ipynb).

<!-- nav -->
[← 8.15 · SystemVerilog Assertions](08_15_SystemVerilog_Assertions.ipynb) · [Home](../../README.md) · [8.17 · SystemVerilog Preprocessor →](08_17_SystemVerilog_Preprocessor.ipynb)